# CS3264 Assignment 1 — Test Your Question Against the Grading Model

This notebook runs the exact model, prompt and decoding settings used to grade
Assignment 1 ("Stump the Model"). Use it to check, before you submit, whether a
question stumps the model.

**Before running anything:** go to `Runtime` → `Change runtime type` → select a
**T4 GPU**, then `Save`. Without a GPU this will be extremely slow.

**How long it takes:** the first run downloads the model (a few GB, a couple of
minutes). After that, each question usually takes a few minutes on a T4; a run
that uses the whole 8192-token budget takes longer. The notebook prints the exact
time and number of tokens for every run.

**What the verdicts mean:**

| Verdict | Meaning |
|---|---|
| `NOT STUMPED` | The model's final answer matched yours within your tolerance. |
| `STUMPED` | The model finished and gave a final number that does not match yours. This is what you want. |
| `STUMPED (no readable number)` | The model finished, but its final line contained no number we could read. Look at the response yourself; in grading a TA checks these cases by hand. |
| `NO RESULT` | The model ran out of its 8192-token budget before finishing, usually because it fell into a repetition loop. This does **not** count as a stump in grading. Rewrite the question until the model commits to a definite answer. |

## 1. Install and import

In [ ]:
# Colab already has torch and a GPU driver. This installs the exact `transformers`
# version used for grading. You can ignore pip's warnings about other packages.
!pip install -q "transformers==5.16.1"

In [ ]:
import time, math, re, json
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime -> Change runtime type -> select a T4 GPU, "
    "then Runtime -> Restart session, and run this notebook again from the top."
)
print("GPU          :", torch.cuda.get_device_name(0))
print("torch        :", torch.__version__, "| CUDA", torch.version.cuda)
print("transformers :", transformers.__version__)

## 2. The exact grading settings

Everything in this cell matches the model, prompt and decoding settings used for
grading. **Do not change anything in this cell** — if you change the model name,
the prompt, or the decoding settings, the notebook no longer tells you anything
about how grading will behave.

The model sees exactly three things: the `SYSTEM_PROMPT`, your question text, and
the `PROMPT_SUFFIX` appended to your question. Nothing else.

In [ ]:
MODEL_ID = "Qwen/Qwen3-1.7B"
MAX_NEW_TOKENS = 8192
ENABLE_THINKING = True
DTYPE = torch.bfloat16
ATTN_IMPLEMENTATION = "sdpa"

SYSTEM_PROMPT = (
    "You are a careful problem solver. Show your work, then give exactly one "
    "final line as instructed."
)
PROMPT_SUFFIX = (
    "\n\nReason step by step, then end your response with a single line in "
    "exactly this format:\nFinal Answer: <number>"
)

# --- answer extraction -------------------------------------------------------
# We take the LAST line of the response that starts with "Final Answer:" and turn
# what follows into one number. Markdown bold, $...$, \boxed{...}, fractions,
# e^{...}, scientific notation and trailing units are handled. A percent sign is
# ignored: "20%" is read as the number 20, so declare such answers the same way.
FINAL_LINE_RE = re.compile(r"Final\s*Answer[\s*_]*:\s*(.+)", re.IGNORECASE)
NUMBER_RE = re.compile(r"[-+]?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?")
FRAC_RE = re.compile(r"^\s*([-+]?\d[\d,]*\.?\d*)\s*/\s*([-+]?\d[\d,]*\.?\d*)\s*$")
DFRAC_RE = re.compile(r"\\[dt]?frac\{([^{}]+)\}\{([^{}]+)\}")
E_POW_RE = re.compile(r"^\s*e\s*\^\s*\{?\s*([-+]?\d[\d,]*\.?\d*)\s*\}?\s*$", re.IGNORECASE)
SCI_RE = re.compile(
    r"^\s*([-+]?\d[\d,]*\.?\d*)\s*(?:[x*\u00d7\u22c5\u00b7]|\\times|\\cdot)\s*10\s*\^\s*\{?\s*([-+]?\d+)\s*\}?\s*$",
    re.IGNORECASE,
)
BOXED_RE = re.compile(r"^\\boxed\{(.+)\}$")
SYMBOLIC_RE = re.compile(r"sqrt|\\pi|\bpi\b|\bln\b|\blog\b|\^|infty|infinit", re.IGNORECASE)


def _normalize(s):
    s = s.replace("\u2212", "-").replace("\u2013", "-")     # unicode minus, en dash
    s = s.replace("**", "").replace("`", "")                # markdown bold / code
    s = s.strip().strip("*_").strip()                       # stray emphasis marks
    s = re.split(r"=|\u2248|\\approx", s)[-1]               # keep RHS of "x = 0.2", "2/3 ≈ 0.667"
    return s.strip().rstrip(".").strip()


def _strip_wrapping(s):
    s = s.strip().strip("$").strip()
    m = BOXED_RE.match(s)
    if m:
        s = m.group(1).strip()
    for a, b in (("\\(", "\\)"), ("\\[", "\\]"), ("<", ">"), ("(", ")")):
        if s.startswith(a) and s.endswith(b) and len(s) > len(a) + len(b) - 1:
            s = s[len(a):-len(b)].strip()
    return s.strip("$").strip()


def _to_float(s):
    try:
        return float(s.strip().replace(",", ""))
    except ValueError:
        return None


def _try_parse_number(s):
    s = s.strip()
    v = _to_float(s)
    if v is not None:
        return v
    m = FRAC_RE.match(s)
    if m:
        num, den = _to_float(m.group(1)), _to_float(m.group(2))
        if num is not None and den not in (None, 0):
            return num / den
    m = E_POW_RE.match(s)
    if m:
        v = _to_float(m.group(1))
        if v is not None:
            return math.exp(v)
    m = SCI_RE.match(s)
    if m:
        v = _to_float(m.group(1))
        if v is not None:
            return v * 10 ** int(m.group(2))
    return None


def extract_answer(text):
    """Return the number on the model's last 'Final Answer:' line, or None."""
    lines = FINAL_LINE_RE.findall(text)
    if not lines:
        return None
    candidate = _strip_wrapping(_normalize(lines[-1]))
    m = DFRAC_RE.search(candidate)
    if m:
        val = _try_parse_number(f"{m.group(1)}/{m.group(2)}")
        if val is not None:
            return val
    val = _try_parse_number(candidate)
    if val is not None:
        return val
    if SYMBOLIC_RE.search(candidate):        # e.g. "\sqrt{2}": not a plain number
        return None
    nums = NUMBER_RE.findall(candidate)
    if len(nums) == 1:                       # e.g. "0.2 units", "20%", "about 0.2"
        return _to_float(nums[0])
    return None                              # several numbers: ambiguous


def is_correct(pred, gold, tol_rel, tol_abs):
    if pred is None or math.isnan(pred):
        return False
    if tol_abs is not None:
        return abs(pred - gold) <= tol_abs
    denom = max(abs(gold), 1e-8)
    return abs(pred - gold) / denom <= tol_rel


print("Settings loaded. Model:", MODEL_ID)

## 3. Load the model

This downloads Qwen3-1.7B (a few GB) the first time you run it. On Colab's free
tier this takes a couple of minutes.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=DTYPE, attn_implementation=ATTN_IMPLEMENTATION)
except TypeError:   # older transformers versions spell the argument torch_dtype
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=DTYPE, attn_implementation=ATTN_IMPLEMENTATION)
model = model.to("cuda").eval()

# Token ids that mark the end of the model's turn. A run that stops on one of
# these finished on its own; a run that does not was cut off by MAX_NEW_TOKENS.
_eos = model.generation_config.eos_token_id
EOS_IDS = set(_eos if isinstance(_eos, (list, tuple)) else [_eos]) | {tokenizer.eos_token_id}
print("Model loaded | dtype", model.dtype, "| end-of-turn token ids", sorted(EOS_IDS))

## 4. Paste your question here

Fill in the four fields below:

- `QUESTION_TEXT` — your question, exactly as you plan to submit it. **Do not**
  include your own "give me the final answer" instruction; the notebook adds the
  same fixed instruction the grading script adds.
- `MY_FINAL_ANSWER` — the correct numeric answer, from your own derivation.
- `MY_TOLERANCE_TYPE` and `MY_TOLERANCE_VALUE` — how close the model's answer must
  be to yours to count as correct (matching `answer_tolerance` in your submission
  file). Use `"relative"` with something like `0.01` (1%) for most questions; use
  `"absolute"` if your answer is 0 or very close to 0.

The pre-filled question is the worked example from the handout. It shows the
format, but it does **not** stump the model: Qwen3-1.7B gets it right. Replace it
with your own question.

In [ ]:
QUESTION_TEXT = (
    "A hard-margin SVM has weight vector w = [6, 8]. The two supporting "
    "hyperplanes are w.x + b = 1 and w.x + b = -1. Report the perpendicular "
    "distance between these two hyperplanes."
)
MY_FINAL_ANSWER = 0.2
MY_TOLERANCE_TYPE = "relative"   # "relative" or "absolute"
MY_TOLERANCE_VALUE = 0.01

## 5. Run it

The cell prints the model's full response (including its `<think>` reasoning),
how long it took, the number we extracted from its final line, and the verdict.

In [ ]:
def build_inputs(question_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question_text + PROMPT_SUFFIX},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
    return tokenizer(text, return_tensors="pt").to(model.device)


def run_question(question_text, final_answer, tolerance_type, tolerance_value, verbose=True):
    tol_rel = tolerance_value if tolerance_type == "relative" else 0.0
    tol_abs = tolerance_value if tolerance_type == "absolute" else None

    inputs = build_inputs(question_text)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,                              # greedy decoding
            temperature=None, top_p=None, top_k=None,     # silence the sampling defaults
            pad_token_id=tokenizer.eos_token_id,
        )
    seconds = time.time() - t0
    generated = out[0, inputs["input_ids"].shape[1]:]
    n_generated = int(generated.shape[0])
    finished = int(generated[-1]) in EOS_IDS
    completion = tokenizer.decode(generated, skip_special_tokens=True)

    model_answer = extract_answer(completion)
    if not finished:
        verdict = "NO RESULT"
    elif model_answer is None:
        verdict = "STUMPED (no readable number)"
    elif is_correct(model_answer, final_answer, tol_rel, tol_abs):
        verdict = "NOT STUMPED"
    else:
        verdict = "STUMPED"

    result = dict(completion=completion, model_answer=model_answer, verdict=verdict,
                  finished=finished, n_generated=n_generated, seconds=seconds)
    if verbose:
        print("=" * 60)
        print("MODEL'S FULL RESPONSE:")
        print("=" * 60)
        print(completion)
        print("=" * 60)
        print(f"Tokens generated               : {n_generated} in {seconds:.0f} s "
              f"({n_generated / max(seconds, 1e-9):.1f} tokens/s)")
        print(f"Model finished on its own?     : {finished}")
        print(f"Model's extracted final answer : {model_answer}")
        print(f"Your declared final answer     : {final_answer}  "
              f"(tolerance: {tolerance_type} {tolerance_value})")
        print("-" * 60)
        print("RESULT:", verdict)
        if verdict == "NO RESULT":
            print("The model hit the 8192-token limit before finishing (usually a repetition loop).")
            print("In grading this does NOT count as a stump. Rewrite the question so the model commits to an answer.")
        elif verdict == "STUMPED (no readable number)":
            print("The model finished, but no number could be read from its final line.")
            print("Read the response above yourself. In grading, a TA checks these cases by hand.")
        elif verdict == "STUMPED":
            print("The model's final answer did not match yours. Run Section 6 to confirm it is deterministic.")
        else:
            print("The model matched your answer, so this question would not earn stump marks.")
        print("=" * 60)
    return result


first_run = run_question(QUESTION_TEXT, MY_FINAL_ANSWER, MY_TOLERANCE_TYPE, MY_TOLERANCE_VALUE)

## 6. Check it is deterministic on this machine

Grading uses greedy decoding, which is deterministic: the same input on the same
hardware and software always gives the same output. This cell runs your question
once more and compares it with Section 5.

What this does and does not prove: if the two runs disagree, something in your
setup is not deterministic and you should not trust the result. If they agree,
the result is reliable **on this kind of machine**. Grading runs on the same type
of GPU as Colab's free tier (an NVIDIA T4) with the same software settings, so
this is the closest preview of grading you can get. A different GPU could, in rare
cases, produce a different reasoning trace.

In [ ]:
second_run = run_question(QUESTION_TEXT, MY_FINAL_ANSWER, MY_TOLERANCE_TYPE, MY_TOLERANCE_VALUE, verbose=False)

print(f"Run 1: {first_run['verdict']:<30} extracted answer = {first_run['model_answer']}")
print(f"Run 2: {second_run['verdict']:<30} extracted answer = {second_run['model_answer']}")
print()
if first_run["completion"] == second_run["completion"]:
    print("Deterministic: both runs produced the identical response.")
elif (first_run["model_answer"] == second_run["model_answer"]
      and first_run["verdict"] == second_run["verdict"]):
    print("Same verdict and answer, although the reasoning text differed slightly. Acceptable.")
else:
    print("NOT reproducible: the two runs disagree. Do not rely on this result.")

## 7. Optional: check your whole submission file

Once you have written all six questions into `<student_id>_submission.jsonl`, run
this cell. It asks you to upload the file (or set `SUBMISSION_PATH` if you have
already uploaded it via the folder icon on the left) and checks the format against
the handout: valid JSON on every line, all required fields, the topic and
failure-mode labels, at least 4 topics with at most 2 questions each, and the
tolerance format.

Set `RUN_MODEL_ON_ALL = True` to also run all six questions through the model.
This takes roughly six times as long as one run in Section 5.

In [ ]:
import os, collections

SUBMISSION_PATH = ""          # e.g. "/content/A0123456X_submission.jsonl"; leave empty to be asked to upload
RUN_MODEL_ON_ALL = False      # True: also run all six questions through the model

CATEGORIES = {"regression_prob", "classification", "pac_learning",
              "nn_backprop", "kernels_svm", "unsupervised"}
FAILURE_MODES = {"formula_confusion", "sign_or_direction", "multi_step_compounding",
                 "edge_case_or_degenerate", "arithmetic_precision", "definition_ambiguity", "other"}
REQUIRED_FIELDS = ["student_id", "question_number", "category", "failure_mode", "question_text",
                   "derivation", "final_answer", "answer_tolerance", "failure_prediction"]
OPTIONAL_FIELDS = ["bonus_note"]


def _is_number(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)


def check_submission(path):
    problems, records = [], []
    if not os.path.exists(path):
        return [f"file not found: {path}"], records
    with open(path, encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                problems.append(f"line {lineno}: not valid JSON ({e.msg} at column {e.colno})")
                continue
            if not isinstance(obj, dict):
                problems.append(f"line {lineno}: must be a JSON object")
                continue
            records.append((lineno, obj))

    if len(records) != 6:
        problems.append(f"expected 6 question lines, found {len(records)}")

    for lineno, r in records:
        tag = f"line {lineno}"
        for k in REQUIRED_FIELDS:
            if k not in r:
                problems.append(f"{tag}: missing field '{k}'")
        for k in r:
            if k not in REQUIRED_FIELDS and k not in OPTIONAL_FIELDS:
                problems.append(f"{tag}: unexpected field '{k}'")
        if "student_id" in r and not re.fullmatch(r"[A-Z]\d{7}[A-Z]", str(r["student_id"])):
            problems.append(f"{tag}: student_id {r['student_id']!r} does not look like a matriculation number such as A0123456X")
        if "category" in r and r["category"] not in CATEGORIES:
            problems.append(f"{tag}: category {r['category']!r} is not one of {sorted(CATEGORIES)}")
        if "failure_mode" in r and r["failure_mode"] not in FAILURE_MODES:
            problems.append(f"{tag}: failure_mode {r['failure_mode']!r} is not one of {sorted(FAILURE_MODES)}")
        qn = r.get("question_number")
        if not (isinstance(qn, int) and not isinstance(qn, bool) and 1 <= qn <= 6):
            problems.append(f"{tag}: question_number must be an integer from 1 to 6")
        for k in ("question_text", "derivation", "failure_prediction"):
            if k in r and (not isinstance(r[k], str) or not r[k].strip()):
                problems.append(f"{tag}: {k!r} must be a non-empty string")
        if isinstance(r.get("question_text"), str) and re.search(r"final answer", r["question_text"], re.IGNORECASE):
            problems.append(f"{tag}: question_text seems to contain its own 'final answer' instruction; remove it (we add one automatically)")
        fa = r.get("final_answer")
        if not _is_number(fa):
            problems.append(f"{tag}: final_answer must be a number, not {type(fa).__name__}")
        tol = r.get("answer_tolerance")
        tol_ok = (isinstance(tol, dict) and tol.get("type") in ("relative", "absolute")
                  and _is_number(tol.get("value")) and tol["value"] > 0)
        if not tol_ok:
            problems.append(f"{tag}: answer_tolerance must look like {{\"type\": \"relative\", \"value\": 0.01}} with value > 0")
        elif tol["type"] == "relative" and _is_number(fa) and abs(fa) < 1e-9:
            problems.append(f"{tag}: final_answer is 0, so answer_tolerance should be absolute, not relative")
        elif tol["type"] == "relative" and tol["value"] > 0.2:
            problems.append(f"{tag}: warning: relative tolerance {tol['value']} is very loose ({tol['value'] * 100:.0f}%)")
        if "bonus_note" in r and (not isinstance(r["bonus_note"], str) or not r["bonus_note"].strip()):
            problems.append(f"{tag}: bonus_note, if present, must be a non-empty string")

    ids = {str(r.get("student_id")) for _, r in records if "student_id" in r}
    if len(ids) > 1:
        problems.append(f"student_id differs between lines: {sorted(ids)}")
    elif len(ids) == 1:
        expected_name = f"{next(iter(ids))}_submission.jsonl"
        if os.path.basename(path) != expected_name:
            problems.append(f"warning: the file should be named {expected_name} (found {os.path.basename(path)})")
    qnums = sorted(r.get("question_number") for _, r in records
                   if isinstance(r.get("question_number"), int) and not isinstance(r.get("question_number"), bool))
    if len(records) == 6 and qnums != [1, 2, 3, 4, 5, 6]:
        problems.append(f"question_number values should be exactly 1 to 6, found {qnums}")
    cats = collections.Counter(r.get("category") for _, r in records if r.get("category") in CATEGORIES)
    if records and len(cats) < 4:
        problems.append(f"only {len(cats)} distinct topic(s) used; at least 4 are required")
    for c, n in cats.items():
        if n > 2:
            problems.append(f"topic {c!r} is used {n} times; at most 2 questions per topic")
    return problems, records


if not SUBMISSION_PATH:
    try:
        from google.colab import files       # upload dialog (Colab only)
        uploaded = files.upload()
        SUBMISSION_PATH = next(iter(uploaded), "")
    except ImportError:
        pass

if not SUBMISSION_PATH:
    print("Set SUBMISSION_PATH to your .jsonl file (or upload it when prompted) and re-run this cell.")
else:
    problems, records = check_submission(SUBMISSION_PATH)
    print(f"Checked {SUBMISSION_PATH}: {len(records)} question line(s) read.")
    if problems:
        print(f"{len(problems)} problem(s) found:")
        for p in problems:
            print(" -", p)
    else:
        print("Format looks good: all checks passed.")
    if RUN_MODEL_ON_ALL:
        if problems:
            print("\nFix the problems above before running the model on the whole file.")
        else:
            print()
            print(f"{'Q#':<3} {'category':<16} {'model answer':>14} {'your answer':>12}  {'verdict':<28} {'tokens':>6} {'secs':>5}")
            for _, r in sorted(records, key=lambda x: x[1]["question_number"]):
                tol = r["answer_tolerance"]
                res = run_question(r["question_text"], r["final_answer"], tol["type"], tol["value"], verbose=False)
                print(f"{r['question_number']:<3} {r['category']:<16} {str(res['model_answer']):>14} "
                      f"{str(r['final_answer']):>12}  {res['verdict']:<28} {res['n_generated']:>6} {res['seconds']:>5.0f}")

## Notes

- This notebook uses the same model, prompt, decoding settings and answer
  extraction as the grading script. It is the closest preview of grading you can get.
- A `STUMPED` result here is a strong signal, but the grading run happens after the
  deadline and is what counts. Always double check that your written `derivation`
  and `final_answer` are correct before relying on any result here: a question with
  a wrong solution earns no stump marks even if the model also gets it wrong.
- A `NO RESULT` run (the model hit the token limit) never counts as a stump. If you
  see one, change the question until the model commits to a definite answer.
- To test another question, edit the fields in Section 4 and re-run Sections 5 and 6.